# 2 · Interactive dashboard **inside your notebook** | 25 minutes
Build visuals directly in Colab/Jupyter with Plotly. No Streamlit server yet. You are querying **the real data you cleaned in Notebook 1**. The instructor or your team should have loaded the team table first.

**Analyst mindset:** Write one finding and one caveat based on your charts.

## Why this notebook matters
A table of 600 rows hides the story; a good chart shows it in seconds. A **dashboard** is a small set of numbers (KPIs) and charts that answers the questions a support manager asks every week: *How many tickets do we have? How fast do we resolve them? Are customers happy? Where are the bottlenecks?* In Notebook 1 we built the clean table; here we **ask it questions and draw the answers**.

## The big picture
![Dashboard workflow: the clean database table feeds SQL aggregates or pandas, Plotly draws a figure, and the analyst writes one finding and one caveat](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb2_dashboard_flow.png)

Every chart in this notebook follows the same loop: **question → numbers → figure → what it says (and what it cannot say)**. Four cells ask you to fill in a small blank (**TODO 1 to 4**); everything else is provided. You will meet six kinds of visuals; each section explains what the chart is for, how the code builds it, and how to read it.

| Step | What we do | Chart |
|---|---|---|
| 0 | Install, connect, import Plotly | none |
| 1 | Load the clean table and look at its columns | none |
| 2 | Ask the database for summary numbers (SQL helpers) | none |
| 3 | The headline numbers | KPI scorecard |
| 4 | Compare ticket types, drill down, see the spread | bubble chart, sunburst, box plot |
| 5 | Explore combinations and switch measures | heatmap, bar chart with dropdown |
| 6 | Checkpoint: one finding and one caveat | none |

## Vocabulary you will meet
| Term | Meaning |
|---|---|
| **KPI** | Key performance indicator: one number that tells you how you are doing (e.g. closed rate). |
| **CSAT** | Customer satisfaction score, here a rating from 1 (bad) to 5 (good). |
| **Categorical column** | Values are labels: `ticket_type`, `channel`, `priority`, `status`. We **group** by these. |
| **Numeric column** | Values are numbers: `satisfaction_rating`, `response_to_resolution_hours`. We **average** or count these. |
| **Mean vs median** | Mean = sum / count, pulled by extreme values. Median = the middle value, robust to extremes. |
| **Outlier** | A value far from the rest, for example one ticket that took 20 hours. |
| **Interactive chart** | You can hover for exact values, zoom, click legend items, and click slices. |

> The numbers quoted below (600 tickets, 34.7 % closed...) come from the instructor's reference run. If your team table is the same sample of 600 tickets you should see the same values.

## Step 0 · Setup
**Install:** the same packages as Notebook 1. `plotly` draws the charts, `psycopg` talks to PostgreSQL, `pandas` holds tables, `python-dotenv` reads your `.env` file. Each notebook installs what it needs so it can run on its own in Google Colab.

In [ ]:
%pip install -q python-dotenv "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

**Connect:** the next cell is the same provided helper as in Notebook 1 (credentials from `.env` or typed in, then `connect()`, `fetch_df()` and `run_sql()`). We repeat it because every notebook is standalone: a new Colab session starts with nothing loaded.

**What you should see:** `Connected to: team_XX` and your database user. If you see an error here, nothing below will work.

### Import the charting tools
- **`plotly.express` (`px`)**: one-line charts (`px.scatter`, `px.box`...). Fast to write; you describe *what* to plot.
- **`plotly.graph_objects` (`go`)**: building blocks (`go.Bar`, `go.Indicator`...). More code, full control; you describe *how* to draw.
- **`pio.renderers.default = "notebook_connected"`**: tells Plotly to load its JavaScript library from the internet (a CDN) so charts also display in Google Colab and Jupyter. That is why you need an internet connection.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## Step 1 · Load your clean table and understand the data
**What:** read **every row** of `team_XX.clean_tickets` into a pandas DataFrame called `data`.

**Why:** some charts (sunburst, box plot, heatmap) are drawn straight from individual rows, so we keep the full table in memory. The `assert` is a safety net: if the table is empty it stops with a clear message instead of drawing empty charts: it means the Notebook 1 load did not happen.

**Note:** this `SELECT *` has no `ORDER BY`, so the rows come back in any order (compare the `ticket_id` column with Notebook 1). That is fine here because we only count, average and group.

In [ ]:
data = fetch_df(f"SELECT * FROM {TEAM_SCHEMA}.clean_tickets")
assert len(data)>0, "Run Notebook 1 ETL first, or ask instructor for recovery."
display(data.head(5)); print("Rows:",len(data))

**What you should see:** the first 5 rows and `Rows: 600`.

### 1.2 · Which columns can we chart?
Before drawing anything, ask: *what kind of column is this, and is it complete?* The cell below counts missing values and distinct values per column.

In [ ]:
display(data.isna().sum().to_frame("missing values").T)
display(data[["ticket_type", "channel", "priority", "status"]].nunique().to_frame("distinct values").T)

**What you should see (reference run):**
- **Missing values:** `response_to_resolution_hours` is missing for most tickets (only **105 of 600** have a value) and `satisfaction_rating` is missing for **392** (so only **208** tickets are rated). That is expected: open and pending tickets have no resolution yet. These gaps matter later because every average is computed **only over the tickets that have a value**.
- **Distinct values:** 5 ticket types, 4 channels, 4 priorities, 3 statuses. Few distinct values = good columns to group by and colour by.

## Step 2 · Two ways to get numbers: SQL or pandas
There are two ways to turn rows into the numbers behind a chart:

| | **Aggregate in SQL** | **Aggregate in pandas** |
|---|---|---|
| How | `GROUP BY` inside the query | `data.groupby(...)` / `pd.crosstab(...)` on the DataFrame |
| Where the work happens | In the database | In your notebook's memory |
| What travels over the network | A **small** result table | **All rows** (we already downloaded them) |
| Best when | The data is big; you only need summaries | The data is small and you want to slice it many ways |

We use **both**: the two helper functions below aggregate in SQL; the sunburst, box plot and heatmap use the DataFrame `data`.

**What the helpers do**
- `metrics_by_type()`: one row per ticket type with the number of tickets, the average resolution hours and the average satisfaction.
- `summary_metrics()`: one row for the **whole table**: total tickets, closed tickets, average satisfaction, average resolution hours.
- Both use `AVG`, which **ignores `NULL`**. That is why we left missing durations as `NULL` in Notebook 1.
- Wrapping a query in a function means we can reuse it, and Notebook 4 (the Streamlit app) uses the same queries.

In [ ]:
def metrics_by_type():
    return fetch_df(f"""
        SELECT ticket_type, COUNT(*) AS tickets,
               ROUND(AVG(response_to_resolution_hours),2) AS avg_response_to_resolution_hours,
               ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction
        FROM {TEAM_SCHEMA}.clean_tickets
        GROUP BY ticket_type ORDER BY tickets DESC
    """)

def summary_metrics():
    return fetch_df(f"""
        SELECT COUNT(*) AS total_tickets,
               COUNT(*) FILTER (WHERE status = 'Closed') AS closed_tickets,
               ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction,
               ROUND(AVG(response_to_resolution_hours),2) AS avg_response_to_resolution_hours
        FROM {TEAM_SCHEMA}.clean_tickets
    """)

In [ ]:
display(metrics_by_type())
display(summary_metrics())

**What you should see:** first a table with 5 rows (one per ticket type, the biggest group is `Technical issue` with 132 tickets), then a single row: 600 tickets, 208 closed, average satisfaction 2.87, average resolution time 7.8 hours. These are the numbers the next charts will draw.

## Step 3 · KPIs: show what matters
A **KPI scorecard** answers "how are we doing overall?" with a few big numbers. Managers look at it first, so each number must be easy to read and honestly labelled.

![Which chart answers which question: KPI scorecard, bubble chart, sunburst, box plot, heatmap and bar chart with dropdown, each with its Plotly function](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb2_chart_chooser.png)

### 3.1 · TODO 1 · Compute the numbers
`summary_metrics().iloc[0]` takes the single result row as a pandas Series. The **closed rate** is not stored in the database, so we compute it in Python: `100 × closed / total`, with a guard so we never divide by zero.

**TODO 1:** fill in the two blanks in the first cell below: the closed rate is `100 × closed tickets ÷ total tickets`. `stats` holds the columns of `summary_metrics()` as labels (`total_tickets`, `closed_tickets`, `avg_satisfaction`, `avg_response_to_resolution_hours`); put the right two labels in the quotes.

In [ ]:
stats = summary_metrics().iloc[0]
# TODO 1: closed rate = 100 * closed tickets / total tickets
closed_rate = (100 * stats["________"] / stats["________"]) if stats["total_tickets"] else 0
print(f"Total tickets: {stats['total_tickets']:,}")
print(f"Closed rate: {closed_rate:.1f}%")
print(f"Average satisfaction: {stats['avg_satisfaction']}")

**What you should see (reference run):** `Total tickets: 600`, `Closed rate: 34.7%`, `Average satisfaction: 2.87`.

### 3.2 · Draw the scorecard
The cell below puts four `go.Indicator` figures side by side. How it works:
- `make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}, ...]])` creates a grid with 4 slots. Indicators are not ordinary x/y charts, so every slot must be declared as type `"indicator"`.
- `mode="number"` shows just a big number; `mode="number+gauge"` adds a dial, with `gauge={"axis": {"range": [...]}}` setting the scale (0 to 100 for a rate, 1 to 5 for a rating).
- `number={"suffix": "%"}` and `valueformat` control how the number is printed; `<b>...</b>` in a title is HTML bold.
- `float(stats[...] or 0)` falls back to 0 if the value is missing (`None`), so the chart never crashes.

In [ ]:
# Dynamic KPI Scorecard
from plotly.subplots import make_subplots

fig_kpi = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]]
)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=int(stats["total_tickets"]),
    title={"text": "<b>Total Tickets</b>", "font": {"size": 15}},
    number={"font": {"color": "#1E3A8A"}}
), row=1, col=1)

fig_kpi.add_trace(go.Indicator(
    mode="number+gauge",
    value=float(closed_rate),
    number={"suffix": "%", "font": {"color": "#047857"}},
    title={"text": "<b>Resolution Rate</b>", "font": {"size": 15}},
    gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#10B981"}}
), row=1, col=2)

fig_kpi.add_trace(go.Indicator(
    mode="number+gauge",
    value=float(stats["avg_satisfaction"] or 0),
    number={"valueformat": ".2f", "font": {"color": "#D97706"}},
    title={"text": "<b>Avg CSAT (1-5)</b>", "font": {"size": 15}},
    gauge={"axis": {"range": [1, 5]}, "bar": {"color": "#F59E0B"}}
), row=1, col=3)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=float(stats["avg_response_to_resolution_hours"] or 0),
    number={"suffix": "h", "valueformat": ".1f", "font": {"color": "#6366F1"}},
    title={"text": "<b>Avg Resolution</b>", "font": {"size": 15}}
), row=1, col=4)

fig_kpi.update_layout(
    height=160,
    margin=dict(l=20, r=20, t=25, b=20),
    paper_bgcolor="#F8FAFC"
)
fig_kpi.show()

**What you should see:** one wide strip with four cards: **Total Tickets 600**, a gauge **Resolution Rate 34.7 %**, a gauge **Avg CSAT 2.87**, **Avg Resolution 7.8 h**.

**Read it honestly (analyst caveats)**
- The card is titled "Resolution Rate", but the number is the share of tickets whose status is `Closed`. It is a **closed rate for this sample, not an official SLA**.
- The CSAT average uses only the **208 rated tickets**, not all 600.
- The resolution time is the time **from first response to resolution**, averaged over the **105 tickets** that have both timestamps. It is not the total ticket lifetime.

💡 **Your turn:** hover over a gauge. Then change the gauge range of the CSAT card from `[1, 5]` to `[0, 5]` and re-run. Does the same 2.87 look better or worse? A chart's scale changes the impression it gives.

## Step 4 · Compare, drill down and look at the spread
Three questions, three charts. For each one: **what we want to know**, **how the code builds it**, **how to read it**.

### 4.1 · Bubble chart: speed versus satisfaction, per ticket type
**Question:** which ticket types are slow, and do they also make customers unhappy?

**Idea:** put the *average resolution time* on the x axis and the *average satisfaction* on the y axis, one bubble per ticket type, bubble **size = number of tickets**. Two dashed lines cut the chart into four zones: the **median time** (vertical) and a **CSAT baseline of 3.0** (horizontal).

![Annotated bubble chart of resolution time versus satisfaction for the five ticket types, split into four zones by the median time and the 3.0 CSAT baseline](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb2_quadrant_anatomy.png)

**How the code builds it**
1. `summary = metrics_by_type()` gives 5 rows, one per ticket type (SQL aggregate from Step 2).
2. `px.scatter(summary, x=..., y=..., size="tickets", color="ticket_type", text="ticket_type")` maps columns to visual properties: position, size, colour, label. `labels={...}` renames the axes for humans.
3. `add_vline` / `add_hline` draw the benchmark lines. The vertical line is the **median of the five averages** (7.8 h), the horizontal 3.0 is a chosen target.

**TODO 2:** in the `px.scatter(...)` call below, the three blanks map columns of `summary` to visual properties: which column goes on the **x axis** (resolution time), which on the **y axis** (satisfaction), and which sets the **bubble size** (number of tickets)? Run `display(summary)` first to see the column names.

In [ ]:
summary = metrics_by_type()

# 1. QUADRANT BUBBLE MATRIX: Resolution Speed vs Satisfaction
# Reveals: Which ticket categories take the longest and hurt CSAT the most
fig_quad = px.scatter(
    summary,
    # TODO 2: map columns to position (x, y) and bubble size
    x="________",
    y="________",
    size="________",
    color="ticket_type",
    text="ticket_type",
    title="<b>1. Support Quadrant Matrix: Resolution Hours vs Customer Satisfaction</b><br><sup>Bubble Size = Ticket Count | Top-Left = Fast & High CSAT | Bottom-Right = Critical Bottlenecks</sup>",
    labels={
        "avg_response_to_resolution_hours": "Average Resolution Time (Hours)",
        "avg_satisfaction": "Average Customer Satisfaction (1-5)",
        "tickets": "Ticket Count"
    },
    template="plotly_white"
)
fig_quad.update_traces(textposition="top center", marker=dict(opacity=0.85, line=dict(width=1.5, color="DarkSlateGrey")))
# Benchmark lines
med_hours = summary["avg_response_to_resolution_hours"].median()
fig_quad.add_vline(x=med_hours, line_dash="dash", line_color="gray", annotation_text=f"Median Time ({med_hours:.1f}h)")
fig_quad.add_hline(y=3.0, line_dash="dash", line_color="red", annotation_text="CSAT Baseline (3.0)")
fig_quad.update_layout(height=480)
fig_quad.show()

**What you should see (reference run):** five bubbles. **Billing inquiry** is the only one above the 3.0 baseline (3.36). **Refund request** is the slowest (9.14 h) with a low rating (2.74). **Product inquiry** is the fastest (6.15 h) but still has the lowest rating (2.68): being fast does not automatically make customers happier. Hover over a bubble for exact values, and click a name in the legend to hide it.

**Caveat:** the ratings come from the 208 rated tickets, a few dozen per type, so small differences between bubbles may be noise.

💡 **Your turn:** change `3.0` in `add_hline` to `2.5` or `3.5`. Which ticket types move between the zones? A baseline is a **decision**, not a fact.

### 4.2 · Sunburst: where do tickets come from, level by level?
**Question:** how do the 600 tickets split by **channel**, then by **priority**, then by **status**?

**Idea:** a sunburst is a pie chart with several rings. The inner ring is the first level, each outer ring splits its parent slice again. Slice size = number of tickets.

![Sunburst with three rings: channel in the centre, priority in the middle ring, status in the outer ring, and the real ticket counts](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb2_sunburst_anatomy.png)

**How the code builds it**
- `px.sunburst(data, path=["channel", "priority", "status"])` takes the **raw rows** and counts them itself. `path` lists the rings from the centre outwards.
- `color="priority"` with `color_discrete_map={...}` gives each priority a fixed colour (red = Critical ... green = Low) so the colours carry meaning.
- No SQL aggregate is needed: this is the pandas route from Step 2.

In [ ]:
# 2. HIERARCHICAL SUNBURST: Interactive Multi-Level Drilldown
# Reveals: Click any slice to zoom into Channel -> Priority -> Status
fig_sun = px.sunburst(
    data,
    path=["channel", "priority", "status"],
    title="<b>2. Interactive Multi-Level Drilldown: Channel → Priority → Status</b><br><sup>Click on any ring segment to zoom in / click center to zoom out</sup>",
    color="priority",
    color_discrete_map={"Critical": "#EF4444", "High": "#F97316", "Medium": "#EAB308", "Low": "#10B981"},
    template="plotly_white"
)
fig_sun.update_layout(height=520, margin=dict(t=50, l=10, r=10, b=10))
fig_sun.show()

**What you should see (reference run):** four inner slices: Chat (164 tickets) is the largest, Phone 152, Email and Social media 142 each. Click the Chat slice to zoom in and see its priorities and statuses; click the centre to zoom out. Across all tickets the status split is Closed 208, Open 191, Pending Customer Response 201.

**Read it honestly:** a sunburst is good for **composition** (how a total splits up) but poor for **precise comparison** of similar slices: the human eye cannot reliably tell 142 from 152. Hover to get the exact count.

### 4.3 · Box plot: averages hide the spread
**Question:** the average resolution time is 7.8 h, but are all tickets close to that? Does priority change the typical time?

**Idea:** a box plot shows the **distribution** of a numeric column for each category, not just its mean.

![Anatomy of a box plot with outliers, whiskers, quartiles and median, next to the median resolution hours per priority](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb2_boxplot_anatomy.png)

**How the code builds it**
- `data.dropna(subset=["response_to_resolution_hours"])` keeps only the tickets that have a duration. An empty value cannot be plotted, and we must not replace it with 0.
- `px.box(x="priority", y="response_to_resolution_hours", points="outliers")` draws one box per priority and shows unusual tickets as dots.
- `category_orders` sets the order Low → Medium → High → Critical (without it Plotly would use the order it finds the values).

**TODO 3:** the first line of the cell below has a blank: which column must **not** be empty for a ticket to appear in this chart? Put its name in the quotes. Remember: we drop missing values, we never replace them with 0.

In [ ]:
# 3. DISTRIBUTION & OUTLIER BOX PLOT: Resolution Time Spread by Priority
# Reveals: Averages hide extreme delays; box plot shows real median and outliers
# TODO 3: keep only the tickets that have a duration
clean_durations = data.dropna(subset=["________"])
fig_box = px.box(
    clean_durations,
    x="priority",
    y="response_to_resolution_hours",
    color="priority",
    points="outliers",
    category_orders={"priority": ["Low", "Medium", "High", "Critical"]},
    title="<b>3. Resolution Time Spread & Outliers by Ticket Priority</b><br><sup>Inspects actual variance and long-tail resolution bottlenecks</sup>",
    labels={"response_to_resolution_hours": "Resolution Time (Hours)", "priority": "Priority"},
    template="plotly_white"
)
fig_box.update_layout(height=450, showlegend=False)
fig_box.show()

**What you should see (reference run):** four boxes. The **median** resolution time is about 9.1 h for Low, 4.1 h for Medium, 7.8 h for High and 6.8 h for Critical priority. The Medium priority also has a **mean** of 5.6 h against a median of 4.1 h: a few slow tickets pull the mean up, which is why we look at the median and the outliers. Hover over a box to read Q1, median and Q3.

**Caveat:** there are only about 25 tickets per priority, so the boxes are **rough**: a surprising result (Low priority being slowest) is a **question to investigate**, not a proven fact.

💡 **Your turn:** change `x="priority"` to `x="channel"` and re-run. Is the spread similar across channels?

## Step 5 · Explore combinations and switch measures
### 5.1 · Heatmap: where do two categories meet?
**Question:** which kinds of issue arrive through which channel?

**Idea:** `pd.crosstab(data["ticket_type"], data["channel"])` builds a **contingency table**: one row per ticket type, one column per channel, each cell = number of tickets. `px.imshow(..., text_auto=True)` colours each cell by its value and prints the number inside it: darker = more tickets.

**TODO 4:** `pd.crosstab(rows, columns)` needs two columns of `data`: put the **ticket type** on the rows and the **channel** on the columns (use the exact column names).

In [ ]:
# 4. CROSS-CHANNEL HEATMAP: Issue Type vs Support Channel
# TODO 4: ticket type on the rows, channel on the columns
channel_cross = pd.crosstab(data["________"], data["________"])
fig_heat = px.imshow(
    channel_cross,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="<b>4. Issue-to-Channel Volume Heatmap</b><br><sup>Where do specific customer issues land? (Darker = Higher concentration)</sup>",
    labels=dict(x="Support Channel", y="Ticket Type", color="Tickets"),
    template="plotly_white"
)
fig_heat.update_layout(height=400)
fig_heat.show()

**What you should see (reference run):** a 5 × 4 grid. The darkest cell is **Technical issue via Phone (42)**; the lightest is **Technical issue via Email (19)**. Each row adds up to that type's total (for example Billing inquiry: 26 + 27 + 29 + 30 = 112). So the type that is the biggest overall also has the most uneven channel mix: worth asking the support team why.

### 5.2 · Bar chart with a dropdown: one chart, three measures
**Question:** how do the channels compare on **volume**, **satisfaction** and **resolution time**, without drawing three separate charts?

**How it works**
1. A SQL query (`GROUP BY channel`) gives one row per channel with three measures.
2. We add **three `go.Bar` traces** to one figure, one per measure. Only the first is `visible=True`.
3. `updatemenus` adds a **dropdown**. Each button has an `args` pair: which traces to show (`{"visible": [True, False, False]}`) and how to change the title and y-axis. Selecting an item runs this update in your browser: **no Python re-run needed**.

In [ ]:
# 5. DYNAMIC INTERACTIVE CHANNEL EXPLORER (Dropdown Metric Switcher)
# Switch between Volume, Satisfaction, and Resolution Time without re-running code!
channel_metrics = fetch_df(f"""
    SELECT channel, 
           COUNT(*) AS tickets,
           ROUND(AVG(satisfaction_rating), 2) AS avg_satisfaction,
           ROUND(AVG(response_to_resolution_hours), 2) AS avg_resolution_hours
    FROM {TEAM_SCHEMA}.clean_tickets
    GROUP BY channel
    ORDER BY tickets DESC
""")

fig_dyn = go.Figure()
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["tickets"],
    name="Ticket Volume",
    text=channel_metrics["tickets"],
    textposition="auto",
    marker_color="#2563EB",
    visible=True
))
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["avg_satisfaction"],
    name="Avg Satisfaction (1-5)",
    text=channel_metrics["avg_satisfaction"],
    textposition="auto",
    marker_color="#059669",
    visible=False
))
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["avg_resolution_hours"],
    name="Avg Resolution Hours",
    text=channel_metrics["avg_resolution_hours"],
    textposition="auto",
    marker_color="#D97706",
    visible=False
))

fig_dyn.update_layout(
    title="<b>5. Dynamic Channel Explorer</b> (Use dropdown below to switch view)",
    template="plotly_white",
    height=450,
    updatemenus=[
        dict(
            active=0,
            buttons=[
                dict(
                    label="📊 Ticket Volume",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "<b>Ticket Volume by Channel</b>", "yaxis": {"title": "Number of Tickets"}}]
                ),
                dict(
                    label="⭐ Customer Satisfaction (CSAT)",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "<b>Average Customer Satisfaction by Channel (1-5)</b>", "yaxis": {"title": "CSAT Rating"}}]
                ),
                dict(
                    label="⏱️ Avg Resolution Hours",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "<b>Average Resolution Hours by Channel</b>", "yaxis": {"title": "Hours"}}]
                ),
            ],
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.22,
            yanchor="top"
        )
    ],
    xaxis_title="Support Channel",
    yaxis_title="Number of Tickets"
)
fig_dyn.show()

**What you should see (reference run):** a bar chart of **ticket volume** per channel (Chat 164, Phone 152, Email and Social media 142) and a dropdown above it. Switch to **Customer Satisfaction**: Chat is highest (3.03) and Phone lowest (2.67). Switch to **Avg Resolution Hours**: Chat is the slowest (8.43 h) and Phone the fastest (7.34 h). So the **fastest** channel (Phone) has the **least satisfied** customers: speed alone does not explain satisfaction, the same message as the bubble chart.

💡 **Your turn:** add a fourth measure to the dropdown (for example the closed rate). You need a new SQL column, a fourth `go.Bar` trace, a fourth `visible` entry in every button, and a fourth button.

### Chart quality checklist
A chart is part of your argument. Before you show it, check:
1. **Title says what the chart shows**, not just "Chart 1".
2. **Axes are labelled with units** (hours, rating 1-5, number of tickets).
3. **Colours mean something** (and are the same for the same thing across charts).
4. **The scale is honest**: a bar chart should start at 0; a different range changes the impression.
5. **Show sample size** or caveats when a group is small.
6. **One message per chart.** If you need three messages, make three charts or use a dropdown.

### Checkpoint
Write down: Which ticket type has the highest volume? Does average satisfaction differ by type? Why are null response-to-resolution values excluded from AVG? Do not label the timestamp-derived metric as contractual SLA compliance.

### What you did, step by step
| Step | Question | Chart | Key idea |
|---|---|---|---|
| 1 | What is in the table? | none | categorical vs numeric columns, missing values |
| 2 | How do I get summary numbers? | none | aggregate in SQL or in pandas |
| 3 | How are we doing overall? | KPI scorecard | a KPI needs an honest label |
| 4.1 | Which types are slow and unhappy? | bubble chart | two measures plus size, benchmark lines |
| 4.2 | How does the total split? | sunburst | composition, drill-down |
| 4.3 | How spread out are the times? | box plot | median and outliers beat the mean |
| 5.1 | Where do two categories meet? | heatmap | crosstab of counts |
| 5.2 | Compare channels on three measures | bar + dropdown | one figure, several views |

### Write one finding and one caveat
A **finding** says what the chart shows, with a number. A **caveat** says what the chart cannot tell us. Example:
- **Finding:** *Billing inquiry is the only ticket type with an average satisfaction above 3.0 (3.36), even though its resolution time (8.45 h) is above the median of 7.8 h.*
- **Caveat:** *Satisfaction comes only from the 208 rated tickets, a few dozen per type, so a difference of a few tenths of a point may be noise, and the chart does not show why customers are happier.*

**Now write yours** (double-click this cell to edit):
- **My finding:** ...
- **My caveat:** ...

### Discuss with your team
1. Why can a KPI such as "average satisfaction" be misleading if many tickets are not rated?
2. Why do we look at the median and the box plot as well as the mean?
3. Which chart in this notebook would you put on a manager's screen first, and why?
4. When would you aggregate in SQL and when in pandas?